# 01 · Exploration de MIT-BIH Arrhythmia

**Exercice 1, étape 1 sur 6.** On ne construit pas encore de modèle. On regarde les données jusqu'à pouvoir répondre à ces questions :

1. Qu'est-ce qu'un enregistrement, concrètement ? (durée, fréquence d'échantillonnage, dérivations, unités)
2. À quoi ressemble un battement normal, et un battement anormal ?
3. Comment les cardiologues ont-ils annoté les battements ? Quels symboles, en quelle quantité ?
4. Les classes sont-elles équilibrées ? Sont-elles réparties uniformément entre les patients ?
5. Quels pièges faudra-t-il éviter au moment du prétraitement et du découpage entraînement / test ?

> Les cellules **« Mes observations »** sont à remplir à la main. C'est là que se fait l'apprentissage : on note ce qu'on voit avant de passer à la suite.

**Prérequis :** `python scripts/download_data.py mitdb` (77 Mo).

## 0. Rappel : ce qu'on regarde

Un ECG mesure l'activité électrique du cœur en millivolts (mV) au cours du temps. Chaque battement laisse trois ondes :

- **P** : les oreillettes se contractent ;
- **QRS** : les ventricules se contractent. C'est le grand pic, et son sommet s'appelle **R** ;
- **T** : les ventricules se rechargent.

MIT-BIH ne contient pas d'images, mais des **signaux** : des suites de nombres mesurés 360 fois par seconde, sur 2 dérivations.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb

DATA = Path("../data/mitdb")
assert (DATA / "RECORDS").exists(), "Lance d'abord : python scripts/download_data.py mitdb"

plt.rcParams["figure.figsize"] = (14, 4)

## 1. La liste des enregistrements

PhysioNet fournit un fichier `RECORDS` qui liste les identifiants. Chaque enregistrement est fait de trois fichiers :

| Fichier | Contenu |
|---|---|
| `100.hea` | en-tête texte : fréquence, nombre d'échantillons, nom des dérivations, âge et sexe |
| `100.dat` | le signal, en binaire |
| `100.atr` | les annotations des cardiologues (position et type de chaque battement) |

In [ ]:
records = (DATA / "RECORDS").read_text().split()
print(len(records), "enregistrements")
print(records)

In [ ]:
# L'en-tête est un simple fichier texte : ouvrons-le tel quel
print((DATA / "100.hea").read_text())

## 2. Lire un enregistrement

`wfdb.rdrecord` lit le signal et le convertit en mV. `p_signal` est un tableau NumPy de forme `(nombre d'échantillons, nombre de dérivations)`.

In [ ]:
rec = wfdb.rdrecord(str(DATA / "100"))

print("Fréquence d'échantillonnage :", rec.fs, "Hz")
print("Dérivations                 :", rec.sig_name)
print("Unités                      :", rec.units)
print("Forme du signal             :", rec.p_signal.shape)
print("Durée                       :", rec.sig_len / rec.fs / 60, "minutes")
print("Commentaires (âge, sexe...) :", rec.comments)

**Exercice rapide :** à 360 Hz, combien d'échantillons y a-t-il dans 10 secondes ? Dans un battement à 75 bpm ?

## 3. Tracer 10 secondes avec les annotations

`wfdb.rdann` lit les annotations : `ann.sample` donne la position (en échantillons) et `ann.symbol` le type de chaque annotation.

In [ ]:
def tracer(record_id, debut_s=0, duree_s=10, derivation=0):
    # Trace duree_s secondes d'un enregistrement avec les symboles d'annotation
    rec = wfdb.rdrecord(str(DATA / record_id))
    ann = wfdb.rdann(str(DATA / record_id), "atr")
    a, b = int(debut_s * rec.fs), int((debut_s + duree_s) * rec.fs)

    t = np.arange(a, b) / rec.fs
    sig = rec.p_signal[a:b, derivation]

    fig, ax = plt.subplots()
    ax.plot(t, sig, lw=0.8)
    for s, sym in zip(ann.sample, ann.symbol):
        if a <= s < b:
            ax.annotate(sym, (s / rec.fs, rec.p_signal[s, derivation]),
                        xytext=(0, 8), textcoords="offset points",
                        ha="center", color="crimson", fontsize=11)
    ax.set(xlabel="temps (s)", ylabel=rec.units[derivation],
           title=f"Enregistrement {record_id}, dérivation {rec.sig_name[derivation]}")
    ax.grid(alpha=0.3)
    plt.show()


tracer("100")

Comparons avec un enregistrement qui contient beaucoup de battements ventriculaires prématurés (symbole `V`) :

In [ ]:
tracer("119", debut_s=0)

### Mes observations (sections 2 et 3)

- Forme d'un battement `N` :
- Forme d'un battement `V` (largeur du QRS, distance avec le battement précédent) :
- Ce qui me surprend :

## 4. Les symboles d'annotation

Toutes les annotations ne sont pas des battements. Certaines marquent un changement de rythme (`+`), du bruit (`~`) ou un artefact (`|`). Voici les principaux **symboles de battement** :

| Symbole | Signification |
|---|---|
| `N` | battement normal |
| `L` / `R` | bloc de branche gauche / droit |
| `A` / `a` / `J` / `S` | battements prématurés supraventriculaires (venant des oreillettes) |
| `V` | contraction ventriculaire prématurée |
| `F` | fusion d'un battement ventriculaire et d'un normal |
| `/` | battement stimulé par un pacemaker |
| `f` | fusion stimulé + normal |
| `Q` | non classable |

Liste complète : [documentation PhysioNet](https://archive.physionet.org/physiobank/annotations.shtml).

In [ ]:
SYMBOLES_BATTEMENT = set("NLRBAaJSVrFejnE/fQ?")

lignes = []
for r in records:
    ann = wfdb.rdann(str(DATA / r), "atr")
    compte = Counter(s for s in ann.symbol if s in SYMBOLES_BATTEMENT)
    compte["record"] = r
    lignes.append(compte)

par_record = pd.DataFrame(lignes).set_index("record").fillna(0).astype(int)
par_record.head()

In [ ]:
total = par_record.sum().sort_values(ascending=False)
print("Nombre total de battements annotés :", total.sum())
total.to_frame("battements").assign(pourcentage=lambda d: (100 * d.battements / d.battements.sum()).round(2))

In [ ]:
ax = total.plot.bar(logy=True)
ax.set(title="Battements par symbole (échelle logarithmique)", ylabel="nombre de battements")
plt.show()

**Pourquoi une échelle logarithmique ?** Essaie sans (`logy=False`) et regarde ce qui arrive aux petites classes.

## 5. Qui contient quoi ? La répartition entre patients

C'est la question la plus importante pour la suite. Un modèle qui voit les battements d'un patient à l'entraînement les reconnaîtra facilement au test, même s'il n'a rien appris de général.

In [ ]:
# Pour chaque symbole : combien d'enregistrements le contiennent, et quelle part vient du plus gros contributeur
resume = pd.DataFrame({
    "nb_records": (par_record > 0).sum(),
    "record_principal": par_record.idxmax(),
    "part_du_principal_%": (100 * par_record.max() / par_record.sum()).round(1),
}).loc[total.index]
resume

In [ ]:
# Carte de chaleur : enregistrements en lignes, symboles en colonnes
fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(np.log1p(par_record[total.index]), aspect="auto", cmap="viridis")
ax.set_xticks(range(len(total)), total.index)
ax.set_yticks(range(len(par_record)), par_record.index, fontsize=7)
ax.set_title("log(1 + nombre de battements) par enregistrement et par symbole")
fig.colorbar(im, ax=ax)
plt.show()

### Mes observations (sections 4 et 5)

- La classe majoritaire représente environ ___ % des battements.
- Les classes rares :
- Certaines classes viennent presque toutes d'un ou deux patients :
- Conséquence si on mélange tous les battements puis qu'on tire 80 % / 20 % au hasard :

## 6. Les patients : âge, sexe, dérivations

L'en-tête contient une ligne de commentaire du type `69 M 1085 1629 x1` : âge, sexe, puis des codes de médicaments et de bande. `-1` signifie âge inconnu.

In [ ]:
infos = []
for r in records:
    h = wfdb.rdheader(str(DATA / r))
    age, sexe = h.comments[0].split()[:2]
    infos.append({"record": r, "age": int(age) if age != "-1" else np.nan,
                  "sexe": sexe, "derivations": " + ".join(h.sig_name)})

patients = pd.DataFrame(infos).set_index("record")
print(patients.sexe.value_counts(), "\n")
print(patients.age.describe().round(1), "\n")
print(patients.derivations.value_counts())

In [ ]:
patients[patients.derivations != "MLII + V1"]

**À retenir pour le prétraitement :**

- La dérivation **MLII** est presque toujours la première, mais pas partout. Dans 102 et 104, elle est absente ; dans 114, elle est en seconde position. Il faut donc sélectionner la dérivation **par son nom**, pas par sa position.
- **201 et 202 viennent du même patient**, d'où 48 enregistrements pour 47 sujets. Ils doivent rester du même côté du découpage.
- Les enregistrements **102, 104, 107 et 217** proviennent de patients porteurs d'un pacemaker (symboles `/` et `f`). La norme AAMI les exclut.

In [ ]:
# Les enregistrements avec pacemaker : vérifions dans les annotations
par_record.loc[["102", "104", "107", "217"], ["/", "f", "N", "V"]]

## 7. Conclusions de l'exploration

À rédiger avant de passer au notebook suivant. Une phrase par point suffit.

1. **Ce qu'est une donnée :**
2. **Déséquilibre des classes :**
3. **Répartition entre patients :**
4. **Pièges identifiés :**
5. **Ce que j'attends comme résultat :** quelle métrique a du sens quand une classe représente la grande majorité des battements ? Pourquoi l'accuracy seule serait trompeuse ?

### Prochaine étape

`02_filtrage_detection_R.ipynb` : filtrer le signal (passe-bande 0,5 à 40 Hz) et détecter les pics R avec Pan-Tompkins.